In [2]:
# Cell 1: imports & paths

import os
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import ipywidgets as widgets

from utils.util_func import get_config
from datasets.point_cloud_datasets.get_dataset import get_point_cloud_dataset
from voxel_models import DLP
from eval.eval_vox import extract_volumes_for_vis

CONFIG_PATH = "./configs/mimicgen_rgb.json"   # <-- your voxel config
CKPT_PATH   = "/home/ellina/Desktop/Code/lpwm-dev/checkpoints_3d/mimicgen_rgb/mimicgen_rgb_best.pt"  # <-- your trained voxel DLP
DEVICE      = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
out = widgets.Output()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Using device: cuda:0


In [3]:
print("CKPT_PATH =", CKPT_PATH)
print("exists:", os.path.exists(CKPT_PATH))
print("isfile:", os.path.isfile(CKPT_PATH))

# list directory contents
!ls -lh "$(dirname $CKPT_PATH)"

# check what kind of file PyTorch is seeing
!file "$CKPT_PATH"

import torch

try:
    obj = torch.load(CKPT_PATH, map_location=DEVICE)
    print("Successfully loaded with torch.load, type:", type(obj))
    if isinstance(obj, dict):
        print("keys:", obj.keys())
except Exception as e:
    print("torch.load failed with:", e)


CKPT_PATH = /home/ellina/Desktop/Code/lpwm-dev/checkpoints_3d/mimicgen_rgb/mimicgen_rgb_best.pt
exists: True
isfile: True
total 263M
-rw-rw-r-- 1 ellina ellina 263M Dec  8 21:34 mimicgen_rgb_best.pt
/home/ellina/Desktop/Code/lpwm-dev/checkpoints_3d/mimicgen_rgb/mimicgen_rgb_best.pt: Zip archive data, at least v?[0] to extract
torch.load failed with: module 'torch' has no attribute '_utils'


In [4]:

# --- User setup ---


# If True -> use voxel/3D model class (DLPVox). If False -> 2D image DLP (DLPImage).
USE_VOX_MODEL = True

# Data
# DATASET_NAME = 'shapes'                       # your dataset key
# DATA_ROOT    = '/path/to/data/root'           # set to your dataset root
MAX_POINTS   = 4096                           # used by get_point_cloud_dataset

# Visualization defaults
VOX_ISO = 0.5          # isosurface threshold for occupancy/density channel
MAX_KP_PLOT = 128      # limit number of keypoints to show

assert os.path.exists(CONFIG_PATH), f'Config not found: {CONFIG_PATH}'
print('[OK] CONFIG_PATH:', CONFIG_PATH)
print('[Info] CKPT_PATH:', CKPT_PATH)
print('[Info] Using model:', 'DLPVox (3D)' if USE_VOX_MODEL else 'DLPImage (2D)')


[OK] CONFIG_PATH: ./configs/mimicgen_rgb.json
[Info] CKPT_PATH: /home/ellina/Desktop/Code/lpwm-dev/checkpoints_3d/mimicgen_rgb/mimicgen_rgb_best.pt
[Info] Using model: DLPVox (3D)


In [5]:
# Cell 2: load config + dataset + model (same hyperparams as train_dlp_pc)

config = get_config(CONFIG_PATH)
print("Loaded config for ds =", config["ds"])

# ---- dataset (voxelized) ----
ds_name        = config['ds']
root           = config['root']
ch             = config['ch']
voxel_grid_whd = config['voxel_grid_whd']
voxel_root     = config.get("voxel_root", None)

dataset = get_point_cloud_dataset(
    ds_name,
    root,
    mode="train",
    voxelize=True,
    voxel_grid_whd=voxel_grid_whd,
    voxel_mode="occupancy",
    cache_dir=voxel_root,
)
print("Dataset length:", len(dataset))

# ---- model hyperparams extracted from config (same as your train code) ----
pad_mode           = config['pad_mode']
n_kp_per_patch     = config['n_kp_per_patch']
n_kp_prior         = config['n_kp_prior']
n_kp_enc           = config['n_kp_enc']
patch_size         = config['patch_size']
anchor_s           = config['anchor_s']

features_dist      = config.get('features_dist', 'gauss')
learned_feature_dim     = config['learned_feature_dim']
learned_bg_feature_dim  = config.get('learned_bg_feature_dim', learned_feature_dim)
n_fg_categories    = config.get('n_fg_categories', 8)
n_fg_classes       = config.get('n_fg_classes', 4)
n_bg_categories    = config.get('n_bg_categories', 4)
n_bg_classes       = config.get('n_bg_classes', 4)

dropout            = config['dropout']
use_resblock       = config['use_resblock']
pint_enc_layers    = config['pint_enc_layers']
pint_enc_heads     = config['pint_enc_heads']

obj_res_from_fc    = config["obj_res_from_fc"]
obj_ch_mult        = config["obj_ch_mult"]
obj_ch_mult_prior  = config.get("obj_ch_mult_prior", obj_ch_mult)
obj_base_ch        = config["obj_base_ch"]
obj_final_cnn_ch   = config["obj_final_cnn_ch"]

bg_res_from_fc     = config["bg_res_from_fc"]
bg_ch_mult         = config["bg_ch_mult"]
bg_base_ch         = config["bg_base_ch"]
bg_final_cnn_ch    = config["bg_final_cnn_ch"]

num_res_blocks     = config["num_res_blocks"]
cnn_mid_blocks     = config.get('cnn_mid_blocks', False)
mlp_hidden_dim     = config.get('mlp_hidden_dim', 256)

scale_std          = config['scale_std']
offset_std         = config['offset_std']
obj_on_alpha       = config['obj_on_alpha']
obj_on_beta        = config['obj_on_beta']

normalize_rgb      = config.get('normalize_rgb', False)

# RGBD extras (just pass what config expects)
separate_depth_features = config.get("separate_depth_features", False)
depth_feature_dim       = config.get("depth_feature_dim", 0)
split_loss              = config.get("split_loss", False)
depth_loss_ratio        = config.get("depth_loss_ratio", 0.0)

# ---- build voxel DLP model ----
model = DLP(
    cdim=ch,
    image_size=voxel_grid_whd[0],           # "jank" same as training
    normalize_rgb=normalize_rgb,

    n_kp_per_patch=n_kp_per_patch,
    patch_size=patch_size,
    anchor_s=anchor_s,
    n_kp_enc=n_kp_enc,
    n_kp_prior=n_kp_prior,

    pad_mode=pad_mode,
    dropout=dropout,

    features_dist=features_dist,
    learned_feature_dim=learned_feature_dim,
    learned_bg_feature_dim=learned_bg_feature_dim,
    n_fg_categories=n_fg_categories,
    n_fg_classes=n_fg_classes,
    n_bg_categories=n_bg_categories,
    n_bg_classes=n_bg_classes,

    scale_std=scale_std,
    offset_std=offset_std,
    obj_on_alpha=obj_on_alpha,
    obj_on_beta=obj_on_beta,

    obj_res_from_fc=obj_res_from_fc,
    obj_ch_mult_prior=obj_ch_mult_prior,
    obj_ch_mult=obj_ch_mult,
    obj_base_ch=obj_base_ch,
    obj_final_cnn_ch=obj_final_cnn_ch,

    bg_res_from_fc=bg_res_from_fc,
    bg_ch_mult=bg_ch_mult,
    bg_base_ch=bg_base_ch,
    bg_final_cnn_ch=bg_final_cnn_ch,

    use_resblock=use_resblock,
    num_res_blocks=num_res_blocks,
    cnn_mid_blocks=cnn_mid_blocks,
    mlp_hidden_dim=mlp_hidden_dim,

    pint_enc_layers=pint_enc_layers,
    pint_enc_heads=pint_enc_heads,

    timestep_horizon=1,

    separate_depth_features=separate_depth_features,
    depth_feature_dim=depth_feature_dim,
    split_loss=split_loss,
    depth_loss_ratio=depth_loss_ratio,
).to(DEVICE)

print(model.info())

# ---- load checkpoint weights (no optimizer) ----
if os.path.exists(CKPT_PATH):
    from utils.log_utils import load_checkpoint
    _ = load_checkpoint(CKPT_PATH, model, optimizer=None, scheduler=None, map_location=DEVICE)
    print(f"Loaded checkpoint from {CKPT_PATH}")
else:
    print("CKPT_PATH: ", CKPT_PATH)
    print("WARNING: CKPT_PATH not found; using random weights.")


Loaded config for ds = voxel
Dataset length: 12800
Creating DLP Prior with cdim:  3
keep top:  80000
KEEP TOP:  80000
Output logvar for feature encoder:  False
cnn_out_shape: torch.Size([32, 8, 8, 8])
IMAGE SIZE:  48
FEATURE MAP EDGE:  6
RESOLUTION:  48


DeferredCudaCallError: CUDA call failed lazily at initialization with error: module 'torch' has no attribute 'version'

CUDA call was originally invoked at:

['  File "/home/ellina/miniconda3/envs/lpwm/lib/python3.8/runpy.py", line 194, in _run_module_as_main\n    return _run_code(code, main_globals, None,\n', '  File "/home/ellina/miniconda3/envs/lpwm/lib/python3.8/runpy.py", line 87, in _run_code\n    exec(code, run_globals)\n', '  File "/home/ellina/miniconda3/envs/lpwm/lib/python3.8/site-packages/ipykernel_launcher.py", line 16, in <module>\n    app.launch_new_instance()\n', '  File "/home/ellina/.local/lib/python3.8/site-packages/traitlets/config/application.py", line 1075, in launch_instance\n    app.start()\n', '  File "/home/ellina/miniconda3/envs/lpwm/lib/python3.8/site-packages/ipykernel/kernelapp.py", line 619, in start\n    self.io_loop.start()\n', '  File "/home/ellina/miniconda3/envs/lpwm/lib/python3.8/site-packages/tornado/platform/asyncio.py", line 199, in start\n    se

In [ ]:
import numpy as np
import torch

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# map kp in [-1,1]^3 to voxel indices
def kp_norm_to_index(kp, D, H, W, kp_range=(-1.0, 1.0)):
    low, high = kp_range
    kp01 = (kp - low) / (high - low)
    x = np.clip(kp01[:, 0] * (W - 1), 0, W - 1)
    y = np.clip(kp01[:, 1] * (H - 1), 0, H - 1)
    z = np.clip(kp01[:, 2] * (D - 1), 0, D - 1)
    return z, y, x


In [ ]:
import torch
import types

state = types.SimpleNamespace()

# pick device from the model
DEVICE = next(model.parameters()).device

@torch.no_grad()
def voxel_forward(idx: int):
    sample = dataset[idx]
    vox = sample["voxels"]
    vox_t = torch.as_tensor(vox, dtype=torch.float32)

    ch = config["ch"]
    if vox_t.ndim == 4 and vox_t.shape[0] == ch:
        vox_t = vox_t.unsqueeze(0)               # [1,C,D,H,W]
    elif vox_t.ndim == 4 and vox_t.shape[-1] == ch:
        vox_t = vox_t.permute(3,0,1,2).unsqueeze(0)
    else:
        raise ValueError(f"Unexpected vox shape {vox_t.shape}")

    vox_t = vox_t.to(DEVICE)

    out = model(
        vox_t,
        warmup=False,
        with_loss=False,
        beta_kl=config["beta_kl"],
        beta_rec=config["beta_rec"],
        kl_balance=config["kl_balance"],
        recon_loss_type=config["recon_loss_type"],
        recon_loss_func=None,
        beta_obj=config.get("beta_obj", 0.0),
    )

    state.out      = out
    state.vox      = vox_t
    state.rec_orig = out["rec"][0].detach().cpu()    # [C,D,H,W]

    # full latent z that decode_all uses (keep on GPU)
    state.z_t_orig = out["z"].detach().clone()       # [1,K,Zdim]

    # numpy copy of just the first 3 dims = keypoint coords
    kp_np = state.z_t_orig[0, :, :3].detach().cpu().numpy()   # [K,3]
    state.kp_orig = kp_np
    state.kp_edit = kp_np.copy()

    print("rec_orig:", state.rec_orig.shape, "z_t_orig:", state.z_t_orig.shape)
    return out


KeyboardInterrupt: 

In [ ]:
@torch.no_grad()
def decode_from_z_edit():
    global z_edit, z_scale_t, z_feat_t, z_obj_on_t, z_depth_t, z_bg_t, z_ctx_t

    if z_edit is None:
        raise RuntimeError("z_edit is None – run voxel_forward(idx) first.")

    z_edit_t = torch.from_numpy(z_edit).unsqueeze(0).to(DEVICE)  # [1,K,3]

    z_scale  = z_scale_t.to(DEVICE)
    z_feat   = z_feat_t.to(DEVICE)
    z_obj_on = z_obj_on_t.to(DEVICE) if z_obj_on_t is not None else None
    z_depth  = z_depth_t.to(DEVICE) if z_depth_t is not None else None
    z_bg     = z_bg_t.to(DEVICE)
    z_ctx    = z_ctx_t.to(DEVICE)

    dec = model.decode_all(
        z_edit_t,
        z_scale,
        z_feat,
        z_obj_on,
        z_depth,
        z_bg,
        z_ctx,
        warmup=False,
    )

    rec = dec["rec"]  # [1,C,D,H,W]
    rec_np = rec[0].detach().cpu().permute(1, 2, 3, 0).numpy()  # [D,H,W,C]
    return rec_np, dec


In [ ]:
import numpy as np
import torch
import plotly.graph_objects as go

def log_rgb_voxels_with_kp_ids(
    name,
    rgb_vol,
    alpha_vol,
    KPx,
    *,
    step=None,
    mode="splat",
    topk=60000,
    alpha_thresh=0.05,
    pad=2.0,
    show_axes=True,
    kp_range=(-1.0, 1.0),
):
    """
    Call your original log_rgb_voxels, then overlay kp indices as text.
    Returns the Plotly figure so Jupyter can render it.
    """
    # 1) call the real function (this builds the fig + logs to wandb if active)
    fig = log_rgb_voxels(
        name=name,
        rgb_vol=rgb_vol,
        alpha_vol=alpha_vol,
        KPx=KPx,
        step=step,
        mode=mode,
        topk=topk,
        alpha_thresh=alpha_thresh,
        mesh_iso=0.2,
        pad=pad,
        show_axes=show_axes,
    )

    # 2) convert KPx to voxel coordinates (same convention as _draw_kp_crosses)
    if isinstance(KPx, torch.Tensor):
        KPx_np = KPx.detach().cpu().numpy()
    else:
        KPx_np = np.asarray(KPx)

    if KPx_np.ndim == 3:
        KPx_np = KPx_np[0]        # [B,K,3] -> [K,3]

    # rgb_vol is [C,D,H,W] or [1,C,D,H,W]
    vol = rgb_vol
    if isinstance(vol, torch.Tensor):
        vol = vol.detach().cpu().numpy()
    if vol.ndim == 5:
        vol = vol[0]
    _, D, H, W = vol.shape

    lo, hi = kp_range  # usually (-1,1)
    kp01 = (KPx_np - lo) / (hi - lo)          # -> [0,1]
    xk = np.clip(kp01[:, 0] * (W - 1), 0, W - 1)
    yk = np.clip(kp01[:, 1] * (H - 1), 0, H - 1)
    zk = np.clip(kp01[:, 2] * (D - 1), 0, D - 1)

    labels = [str(i) for i in range(KPx_np.shape[0])]

    # 3) add a text-only scatter trace so you see the index next to each cube
    fig.add_trace(go.Scatter3d(
        x=xk,
        y=yk,
        z=zk,
        mode="text",
        text=labels,
        textposition="top center",
        name="kp_idx",
        showlegend=True,
    ))

    return fig


In [ ]:
from eval.eval_vox import log_rgb_voxels
voxel_forward(1)  # fills rec_vol, mu_tot, etc.

fig = log_rgb_voxels_with_kp_ids(
    name="nb/rec_rgb_splat_kp_ids",
    rgb_vol=rec_vol,
    alpha_vol=None,
    KPx=mu_tot,
    step=None,
    mode="splat",
    topk=60000,
    alpha_thresh=0.05,
    pad=2.0,
    show_axes=True,
    kp_range=model.kp_range,   # or (-1.0, 1.0)
)

fig  # last line so Jupyter shows it


KP:  torch.Size([1, 64, 3])
cov:  torch.Size([1, 64, 3, 3])
MU 0:  tensor([-0.1518,  0.0536,  0.0866], device='cuda:0')
FILTERING!!!!!!!!!!!!
ParticleFeatEnc x shape: torch.Size([1, 3, 64, 64, 64]) min/max per channel: [(0.0, 1.0), (0.0, 1.0), (0.0, 1.0)]
ParticleFeatEnc x shape: torch.Size([1, 3, 64, 64, 64]) min/max per channel: [(0.0, 1.0), (0.0, 1.0), (0.0, 1.0)]
max |corr(feature, R/G/B)| = [0.5779885649681091, 0.6086763143539429, 0.5923290252685547]
Z FEAT SHAPE:  torch.Size([1, 32, 8, 8, 8])
[ObjectDecoderCNN] y stats: min=-24.2785 max=2.0258 mean=-8.4834 std=3.5470
  ch0: min=-15.3781 max=1.7393 mean=-6.6066 std=2.1356
  ch1: min=-24.2785 max=2.0233 mean=-9.0402 std=3.7052
  ch2: min=-23.9102 max=2.0258 mean=-9.0363 std=3.6931
  ch3: min=-24.0708 max=2.0210 mean=-9.2504 std=3.6976
[decode_rgb_unified] patches_t shape: torch.Size([1, 24, 4, 64, 64, 64])
  alpha_logits: min=-15.3780 max=1.6721 mean=-8.0402 std=2.8092
  alpha_prob:   min=0.0000 max=0.8419 mean=0.0090 std=0.0315
  

In [ ]:
import numpy as np
import torch

import numpy as np


def move_kp_indices(indices, dx=0.0, dy=0.0, dz=0.0, clip=True):
    """
    Edit the numpy kp array state.kp_edit (shape [K,3]).
    indices: int or list/array of ints.
    """
    if not hasattr(state, "kp_edit") or state.kp_edit is None:
        raise RuntimeError("kp_edit is None – run voxel_forward(idx) first.")

    kp = state.kp_edit        # [K,3] np
    K  = kp.shape[0]

    idx_arr = np.array(indices, dtype=int).ravel()
    valid   = (idx_arr >= 0) & (idx_arr < K)
    idx_v   = idx_arr[valid]

    if idx_v.size == 0:
        print(f"[move_kp_indices] no valid indices in {indices} (K={K})")
        return

    if (~valid).any():
        print(f"[move_kp_indices] ignoring out-of-range:",
              idx_arr[~valid].tolist(), f"(K={K})")

    kp[idx_v, 0] += float(dx)
    kp[idx_v, 1] += float(dy)
    kp[idx_v, 2] += float(dz)

    if clip:
        lo, hi = getattr(model, "kp_range", (-1.0, 1.0))
        kp[idx_v] = np.clip(kp[idx_v], lo, hi)

    state.kp_edit = kp  # store back (same object, but explicit)


def reset_kp(idx):
    """Reset a single kp back to its original value."""
    global z_orig, z_edit
    if z_orig is None or z_edit is None:
        raise RuntimeError("Run voxel_forward(...) first.")
    if not (0 <= idx < z_edit.shape[0]):
        raise IndexError(f"kp index {idx} out of range [0, {z_edit.shape[0]-1}]")
    z_edit[idx] = z_orig[idx]
    return z_edit[idx].copy()

def reset_all_kps():
    """Reset all kps to original."""
    global z_orig, z_edit
    if z_orig is None:
        raise RuntimeError("Run voxel_forward(...) first.")
    z_edit = z_orig.copy()


In [ ]:
@torch.no_grad()
def decode_from_kp_edit():
    """
    Copy kp_edit back into the first 3 dims of z, then call decode_all.
    """
    if not hasattr(state, "z_t_orig"):
        raise RuntimeError("z_t_orig not set – run voxel_forward(idx) first.")
    if not hasattr(state, "kp_edit") or state.kp_edit is None:
        raise RuntimeError("kp_edit not set – run voxel_forward(idx) first.")

    # 1) start from original z
    z_edit = state.z_t_orig.clone().to(DEVICE)      # [1,K,Zdim]

    # 2) overwrite first 3 dims with edited coords
    kp_t = torch.from_numpy(state.kp_edit).to(DEVICE)   # [K,3]
    z_edit[0, :, :3] = kp_t

    # 3) use the same other latents as in forward
    out   = state.out
    z_scale    = out["z_scale"].to(DEVICE)
    z_features = out["z_features"].to(DEVICE)
    z_obj_on   = out.get("obj_on", None)
    if z_obj_on is not None:
        z_obj_on = z_obj_on.to(DEVICE)
    z_depth    = out.get("z_depth", None)
    if z_depth is not None:
        z_depth = z_depth.to(DEVICE)
    z_bg       = out["z_bg_features"].to(DEVICE)
    z_ctx      = out.get("z_context", None)
    if z_ctx is not None:
        z_ctx = z_ctx.to(DEVICE)

    dec = model.decode_all(
        z_edit, z_scale, z_features,
        z_obj_on, z_depth, z_bg, z_ctx,
        warmup=False,
    )

    rec_edit = dec["rec"][0].detach().cpu()   # [C,D,H,W]
    state.rec_edit = rec_edit
    return rec_edit, dec


In [ ]:
print("z_orig shape:", None if z_orig is None else z_orig.shape)
print("z_edit shape:", None if z_edit is None else z_edit.shape)
print("mu_tot shape:", None if 'mu_tot' not in globals() else mu_tot.shape)


z_orig shape: (1, 24, 3)
z_edit shape: (1, 24, 3)
mu_tot shape: torch.Size([1, 24, 3])


In [ ]:
move_kp_indices([10, 3, 9, 18, 1, 12], dx=0.1, dy=-0.1)
rec_edit, dec = decode_from_kp_edit()

fig_edit = log_rgb_voxels_with_kp_ids(
    name="nb/rec_edit_multi",
    rgb_vol=rec_edit,
    alpha_vol=None,
    KPx=state.kp_edit,     # edited kp positions
    step=None,
    mode="splat",
    alpha_thresh=0.05,
    pad=2.0,
    show_axes=True,
    kp_range=model.kp_range,
)
fig_edit


z edit shape: torch.Size([1, 1, 24, 3])


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
